In [1]:
import random
import numpy as np 

# Генерация биматричной игры
def generate_bimatrix_game(rows, cols, min_val, max_val):
    matrix_game = []
    for row in range(0,rows):
        matrix_row = []
        for col in range(0,cols):
            matrix_row.append( (random.randint(min_val,max_val), random.randint(min_val,max_val)) )
        matrix_game.append(matrix_row)
    
    return matrix_game

# Известные игры
def crossroad_game():
    matrix_game = [
        [(1,1), (1,2)],
        [(2,1), (0,0)]
    ]
    return matrix_game
def family_argue_game():
    matrix_game = [
        [(4,1), (0,0)],
        [(0,0), (1,4)]
    ]
    return matrix_game
def criminal_game():
    matrix_game = [
        [(-5,-5), (0,-10)],
        [(-10,0), (-1,-1)]
    ]
    return matrix_game

# Игра из 16 варианта
def game_var_16():
    matrix_game = [
        [(8,7), (1,2)],
        [(2,0), (3,4)]
    ]
    return matrix_game
def game_test():
    matrix_game = [
        [(0,1), (11,4)],
        [(7,8), (6,3)]
    ]
    return matrix_game

# Красивый вывод биматричной игры
def print_bimatrix_game(matrix_game):    
    rows = matrix_game.__len__()
    cols = matrix_game[0].__len__()
    
    # Находим максимальную длину для каждого столбца
    col_widths = []
    for col in range(cols):
        max_width = 0
        for row in range(rows):
            pair_str = f"({matrix_game[row][col][0]}, {matrix_game[row][col][1]})"
            max_width = max(max_width, len(pair_str))
        col_widths.append(max_width)
    
    # Выводим матрицу с выравниванием по левому краю
    for row in range(rows):
        for col in range(cols):
            pair_str = f"({matrix_game[row][col][0]}, {matrix_game[row][col][1]})"
            if col == 0:
                print(f"[ {pair_str:<{col_widths[col]}}", end=" ")
            elif col == cols - 1:
                print(f"{pair_str:<{col_widths[col]}} ]")
            else:
                print(f"{pair_str:<{col_widths[col]}}", end=" ")

# Поиск ситуаций, равновесных по Нэшу
def find_Nash(game_matrix):
    rows = game_matrix.__len__()
    cols = game_matrix[0].__len__()
    nash_equilibria = []
    
    for row in range(rows):
        for col in range(cols):
            current_payoff1, current_payoff2 = game_matrix[row][col]
            is_nash = True
            
            # Проверяем, может ли игрок 1 улучшить свой выигрыш, изменив стратегию
            for new_row in range(rows):
                if new_row != row:
                    payoff1 = game_matrix[new_row][col][0]
                    if payoff1 > current_payoff1:
                        is_nash = False
                        break
            
            if not is_nash:
                continue

            # Проверяем, может ли игрок 2 улучшить свой выигрыш, изменив стратегию
            for new_col in range(cols):
                if new_col != col:
                    payoff2 = game_matrix[row][new_col][1]
                    if payoff2 > current_payoff2:
                        is_nash = False
                        break
        
            if is_nash:
                nash_equilibria.append((row, col))
    
    return nash_equilibria

# Поиск ситуаций, оптимальных по Парето
def find_Pareto(game_matrix):
    rows = game_matrix.__len__()
    cols = game_matrix[0].__len__()
    pareto_optimal = []
    
    # Собираем все клетки
    all_cells = [(row, col) for row in range(rows) for col in range(cols)]
    
    for row, col in all_cells:
        payoff1, payoff2 = game_matrix[row][col]
        is_pareto_optimal = True
        
        # Проверяем, нет ли клетки, которая доминирует текущую
        for new_row, new_col in all_cells:
            if (new_row, new_col) == (row, col):
                continue
            
            other_payoff1, other_payoff2 = game_matrix[new_row][new_col]
            
            # Если другая клетка не хуже по всем показателям и лучше хотя бы по одному
            if (other_payoff1 >= payoff1 and other_payoff2 >= payoff2 and
                (other_payoff1 > payoff1 or other_payoff2 > payoff2)):
                is_pareto_optimal = False
                break
        
        if is_pareto_optimal:
            pareto_optimal.append((row, col))
    
    return pareto_optimal

# Поиск ситуации равновесия в смешанных стратегиях
def find_Nash_mixed(game_matrix):
    A_matrix = []
    B_matrix = []
    for game_matrix_row in game_matrix:
        A_matrix_row = []
        B_matrix_row = []
        for element in game_matrix_row:
            A_matrix_row.append(element[0])
            B_matrix_row.append(element[1])
        A_matrix.append(A_matrix_row)
        B_matrix.append(B_matrix_row)
    
    u_vector = np.ones(A_matrix.__len__())
    A_matrix_inverse = np.linalg.inv(A_matrix)
    B_matrix_inverse = np.linalg.inv(B_matrix)

    print(np.dot(u_vector, B_matrix_inverse))

    v1 = 1. / ( np.dot( np.dot(u_vector, A_matrix_inverse), u_vector.T ) )
    v2 = 1. / ( np.dot( np.dot(u_vector, B_matrix_inverse), u_vector.T ) )

    x_vector = v2 * np.dot(u_vector, B_matrix_inverse)
    y_vector = v1 * np.dot(A_matrix_inverse, u_vector.T)

    return x_vector, y_vector, v1, v2

if __name__ == "__main__":
    game_matrix = generate_bimatrix_game(10,10,-10,10)

    # Генерация матрицы
    print("Сгенерированная матрица:")
    print_bimatrix_game(game_matrix)
    print()

    # Поиск по Нэшу
    nash_indexes = find_Nash(game_matrix)
    print("Равновесные по Нэшу ситуации:")
    if nash_indexes:
        for indexes in nash_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()

    # Поиск по Парето
    pareto_indexes = find_Pareto(game_matrix)
    print("Оптимальные по Парето ситуации:")
    if pareto_indexes:
        for indexes in pareto_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()

    # Пересечения
    print("Пересечения:")
    found = False
    if nash_indexes and pareto_indexes:
        for indexes_1 in nash_indexes:
            for indexes_2 in pareto_indexes:
                if indexes_1 == indexes_2:
                    print(f"Индексы: {indexes_1}\tЗначения: {game_matrix[indexes_1[0]][indexes_1[1]]}")
                    found = True
    if not found:
        print("Не найдены")
    print()
    print()

    # Проверка на известных играх
    # Перекресток
    print("Перекресток:")
    game_matrix = crossroad_game()
    nash_indexes = find_Nash(game_matrix)
    pareto_indexes = find_Pareto(game_matrix)
    print("Равновесные по Нэшу ситуации:")
    if nash_indexes:
        for indexes in nash_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()
    print("Оптимальные по Парето ситуации:")
    if pareto_indexes:
        for indexes in pareto_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()
    print("Пересечения:")
    found = False
    if nash_indexes and pareto_indexes:
        for indexes_1 in nash_indexes:
            for indexes_2 in pareto_indexes:
                if indexes_1 == indexes_2:
                    print(f"Индексы: {indexes_1}\tЗначения: {game_matrix[indexes_1[0]][indexes_1[1]]}")
                    found = True
    if not found:
        print("Не найдены")
    print()
    print()

    # Семейный спор
    print("Семейный спор:")
    game_matrix = family_argue_game()
    nash_indexes = find_Nash(game_matrix)
    pareto_indexes = find_Pareto(game_matrix)
    print("Равновесные по Нэшу ситуации:")
    if nash_indexes:
        for indexes in nash_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()
    print("Оптимальные по Парето ситуации:")
    if pareto_indexes:
        for indexes in pareto_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()
    print("Пересечения:")
    found = False
    if nash_indexes and pareto_indexes:
        for indexes_1 in nash_indexes:
            for indexes_2 in pareto_indexes:
                if indexes_1 == indexes_2:
                    print(f"Индексы: {indexes_1}\tЗначения: {game_matrix[indexes_1[0]][indexes_1[1]]}")
                    found = True
    if not found:
        print("Не найдены")
    print()
    print()

    # Дилемма заключенного
    print("Дилемма заключенного:")
    game_matrix = criminal_game()
    nash_indexes = find_Nash(game_matrix)
    pareto_indexes = find_Pareto(game_matrix)
    print("Равновесные по Нэшу ситуации:")
    if nash_indexes:
        for indexes in nash_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()
    print("Оптимальные по Парето ситуации:")
    if pareto_indexes:
        for indexes in pareto_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    print()
    print("Пересечения:")
    found = False
    if nash_indexes and pareto_indexes:
        for indexes_1 in nash_indexes:
            for indexes_2 in pareto_indexes:
                if indexes_1 == indexes_2:
                    print(f"Индексы: {indexes_1}\tЗначения: {game_matrix[indexes_1[0]][indexes_1[1]]}")
                    found = True
    if not found:
        print("Не найдены")
    print()
    print()


    # Часть 2 (2x2)
    print("Часть 2 - матрица 2x2")
    game_matrix = game_var_16()
    # game_matrix = game_test()
    print_bimatrix_game(game_matrix)
    nash_indexes = find_Nash(game_matrix)
    print("Равновесные по Нэшу ситуации:")
    if nash_indexes:
        for indexes in nash_indexes:
            print(f"Индексы: {indexes}\tЗначения: {game_matrix[indexes[0]][indexes[1]]}")
    else:
        print("Не найдены")
    
    # Смешанная ситуация равновесия в дополнениях
    if nash_indexes.__len__() != 1:
        print("Смешанная ситуация равновесия в дополнениях:")
        x_vector, y_vector, v1, v2 = find_Nash_mixed(game_matrix)
        print(f"Вектор x: {x_vector}, выигрыш: {v1}")
        print(f"Вектор y: {y_vector}, выигрыш: {v2}")
    else:
        print("Смешанной ситуации равновесия в дополнениях не существует")
    


Сгенерированная матрица:
[ (-10, 9)  (6, -8)   (-4, 4)   (5, -8)  (8, 1)   (-8, -9) (0, 7)   (5, -1)   (-7, 7)  (-7, -1)  ]
[ (9, 0)    (8, 4)    (-9, 3)   (4, -9)  (-6, -9) (10, -6) (-6, -8) (-10, 0)  (-5, 10) (-8, -10) ]
[ (6, 1)    (2, 9)    (-7, -5)  (7, 6)   (10, -6) (-7, -6) (1, 6)   (-7, -5)  (4, -4)  (7, -4)   ]
[ (1, 6)    (10, -8)  (-8, -5)  (8, 9)   (6, 8)   (-3, -6) (-4, 10) (10, -5)  (-9, -5) (-6, -8)  ]
[ (-10, -6) (-5, 1)   (-7, -8)  (9, 3)   (-9, 6)  (-3, 8)  (-2, 1)  (-5, 10)  (-2, 6)  (2, 9)    ]
[ (-9, -7)  (-1, -10) (7, 7)    (10, -8) (-2, -5) (-2, 0)  (5, 8)   (3, 7)    (-8, 10) (7, 9)    ]
[ (5, 5)    (0, 8)    (-3, 9)   (0, 9)   (-9, 7)  (-3, -3) (-2, 7)  (-9, -10) (3, 6)   (6, 8)    ]
[ (-2, 6)   (7, 7)    (-10, -9) (3, 7)   (2, 4)   (2, -8)  (-8, -1) (6, 10)   (0, -2)  (-7, -8)  ]
[ (4, -3)   (-4, -3)  (3, -5)   (9, 7)   (2, -2)  (-3, -4) (6, -10) (-9, 8)   (-3, -9) (10, 10)  ]
[ (2, 9)    (9, -8)   (-1, 4)   (-9, -1) (-2, -4) (3, 2)   (6, -1)  (7, 8)    (2, 10